## Паплайн для загрузки данных, добавления истинных значений и расчета метрик

В данном проекте:
- загрузка платежй из БД, начиная с id платежа после последнего обучения модели (могут быть новые платежи, но календарно старые - если придет новый клиент и загрузит старые выписки - эти платежи будут полезны для оценки деградации модели по пункту 1 ниже),
- отсеивание платежей с отсутствующими дополнительными колонками(статья+проект+категория донора) - без них класификация не сработает,
- разделение платежей со статьями по имеющемуся master-словарю(блок 1) и staging-словарю с новыми статьями(блок 2),
- расчет метрик:
  
    1. - добавление истинных меток по master-словарю статей, 
       - расчет метрик прогноза - смотрим деградацию модели по старым статьям, которые были в обучении, метрика должна быть очень высокой, потому что модель работает фактически как эвристика, старые размеченные вручную статьи дают очень точный сигнал даже для новых платежей;
  
    2. - разметка нового staging-словаря (эксперт/фидбэк клиентов),
       - добавление истинных значений классификации для платежей с новыми статьями. Они могут быть близкими по значению или с опечатками относительно старых - это позволит оценить обобщающую способность модели,
       - расчет метрик по платежам с новыми статьями.  


In [32]:
import pandas as pd
import numpy as np
import random
import os
import requests

# ⏳ прогресс-бары
from tqdm import tqdm

# загрузка моделей и функци для предобработки данных
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report,roc_auc_score

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.width', 0)
pd.set_option('display.max_colwidth', None)

os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [7]:
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

### Загрузим и подготовим данные для прогноза

In [8]:
# загружаем все платежи

url_down = "https://api.lemonpie.tech/api/payments/ai"
headers = {"Authorization": "Bearer YOUR_API_TOKEN"}

#start_date = str((pd.Timestamp.today().normalize() - pd.Timedelta(days=1)).date())
#end_date = start_date #str(pd.Timestamp.today().normalize().date())
#start_date = "" #"2025-10-10"
#end_date = "" #"2025-10-20"
start_date = "2025-09-15"
end_date = "2025-10-15"


params = {
    "limit": 5000,
    "page": 1,
    "expenditure": "incoming"
    #"date_from": start_date,
    #"date_to": end_date  
}

all_data = []

with tqdm(desc="Загружено страниц", unit=" стр", dynamic_ncols=True) as pbar:
    while True:
        response = requests.get(url_down, headers=headers, params=params)
        if response.status_code != 200:
            print(f"❌ Ошибка загрузки данных с сервера: {response.status_code}")
            break

        result = response.json()
        page_data = result.get("data", [])
        if not page_data:
            break
        
        all_data.extend(page_data)

        params["page"] += 1
        pbar.update(1)
        
# преобразуем в таблицу (вложенные поля будут с __)
data_full = pd.json_normalize(all_data, sep="__")
print(f"ℹ️ Данные загружены с сервера. Количество записей: {len(data_full)}")
data_full.to_csv("data_download.csv", index=False)

Загружено страниц: 147 стр [19:30,  7.96s/ стр]


ℹ️ Данные загружены с сервера. Количество записей: 734586


In [9]:
# отсеиваем платежи, участвовавшие в последнем обучении, по id
train_history = pd.read_csv('train_history.csv')
last_train_payment_id = train_history['last_payment_id'].iloc[-1]

display(last_train_payment_id)

data_new = data_full[data_full['id']>last_train_payment_id].copy()

data_full['date'] = pd.to_datetime(data_full['date'], errors='coerce')

#data_new = data_full[(data_full['id'] > last_train_payment_id) & (data_full['date'] < pd.Timestamp('2025-12-04'))].copy()

display(data_new.id.count(),data_new.uc__uc_id.isna().sum())

1206719

37313

6444

In [10]:
print('Проверим пропуски по основным признакам:\n',
    (data_new[['purpose','articles__name','projects__name','counterparties__name']].isna().sum()))
print(
    "Количество строк, где все 3 дополнительных признака отсутствуют:",
    data_new[['articles__name','projects__name','counterparties__name']]
        .isna()
        .all(axis=1)
        .sum()
)

Проверим пропуски по основным признакам:
 purpose                  163
articles__name          5525
projects__name          8165
counterparties__name    6159
dtype: int64
Количество строк, где все 3 дополнительных признака отсутствуют: 5440


In [11]:
data_new = data_new.dropna(subset=['articles__name', 'projects__name', 'counterparties__name'], how='all') # здесь нет прогнозов изза отсутствия основных признаков
data_new = data_new.dropna(subset=['uc__uc_id']) # здесь нет прогнозов каким-то причинам
data_new = data_new.dropna(subset=['articles__name']) #сбраываем строки где нет значения статьи, мы не сможем истинные значения расставаить по этим строкам в соответствии со словарем
data_new.id.count()

30836

Разделим данные в соответствии со значениями поля статья по master-словарю (который использовался при обучении)

In [12]:
main_dict = pd.read_csv('main_dict.csv')

In [13]:
display(main_dict.raw.count(),data_full.articles__name.nunique())

813

857

In [14]:
data_new['articles__name'] = data_new['articles__name'].str.lower()

data_new_main_articles = data_new.merge(
    main_dict[['raw', 'uc_id']],
    left_on='articles__name',
    right_on='raw',
    how='left'
)

data_new_main_articles.rename(columns={'uc_id': 'target'}, inplace=True)
data_new_main_articles.drop(columns=['raw'], inplace=True)
data_new_main_articles.head(3)

,id,account_id,contractor_id,date,payments_amount,purpose,article_id,expenditure,project_id,counterpartie_id,donor_id,robot_id,donor_cat_id,accounts__id,accounts__user_id,articles__id,articles__user_id,articles__parent_id,articles__name,projects__id,projects__user_id,projects__parent_id,projects__name,counterparties__id,counterparties__user_id,counterparties__parent_id,counterparties__name,robots__id,robots__user_id,article_alternative_names__user_id,uc__uc_id,target
0,1206722,1446,1807,2025-12-21,50.0,Зачисление по QR коду ID AS1A000I2JFB11GR9TFQ8VHOA1NS693M от 21.12.2025. НДС не предусмотрен.,35155,incoming,1831,0,0,4599,6228,1446.0,114.0,35155.0,114.0,28282.0,точка qr,1831.0,114.0,1829.0,Кошкин дом,6228.0,114.0,6226.0,... массовые,4599.0,114.0,NaN,1.0,1.0
1,1206724,1446,1807,2025-12-21,1.0,Зачисление по QR коду ID AS1A000I2JFB11GR9TFQ8VHOA1NS693M от 21.12.2025. НДС не предусмотрен.,35155,incoming,1831,0,0,4599,6228,1446.0,114.0,35155.0,114.0,28282.0,точка qr,1831.0,114.0,1829.0,Кошкин дом,6228.0,114.0,6226.0,... массовые,4599.0,114.0,NaN,1.0,1.0
2,1206725,1446,1807,2025-12-21,500.0,Зачисление по QR коду ID AS1A000I2JFB11GR9TFQ8VHOA1NS693M от 21.12.2025. НДС не предусмотрен.,35155,incoming,1831,0,0,4599,6228,1446.0,114.0,35155.0,114.0,28282.0,точка qr,1831.0,114.0,1829.0,Кошкин дом,6228.0,114.0,6226.0,... массовые,4599.0,114.0,NaN,1.0,1.0


In [15]:
data_new_staging_articles = data_new_main_articles[data_new_main_articles.target.isna()].copy()
data_new_main_articles = data_new_main_articles.dropna(subset=['target'])

display(data_new_main_articles.id.count(),data_new_staging_articles.id.count())

27419

3417

In [16]:
# считаем метрики прогнозов по статьям из основного словаря статей (который участвовал в обучении модели) 

accuracy = accuracy_score(data_new_main_articles['target'], data_new_main_articles['uc__uc_id'])
precision = precision_score(data_new_main_articles['target'], data_new_main_articles['uc__uc_id'], average='weighted')
recall = recall_score(data_new_main_articles['target'], data_new_main_articles['uc__uc_id'], average='macro')
f1 = f1_score(data_new_main_articles['target'], data_new_main_articles['uc__uc_id'], average='weighted')

print("Accuracy:", accuracy)
print("Precision (weighted):", precision)
print("Recall (macro):", recall)
print("F1-score (weighted):", f1)

print("\nОтчет по классам:")
print(classification_report(data_new_main_articles['target'], data_new_main_articles['uc__uc_id']))

Accuracy: 0.9928881432583245
Precision (weighted): 0.9929405252490546
Recall (macro): 0.9873175056371934
F1-score (weighted): 0.9928971677853334

Отчет по классам:
              precision    recall  f1-score   support

         1.0       1.00      1.00      1.00     18324
         2.0       0.99      0.99      0.99      5639
         3.0       0.92      0.97      0.95       326
         4.0       0.99      1.00      1.00       833
         5.0       0.99      0.99      0.99       643
         6.0       0.99      0.96      0.98       975
         7.0       0.97      0.99      0.98       254
         8.0       0.94      0.98      0.96        60
         9.0       1.00      1.00      1.00       365

    accuracy                           0.99     27419
   macro avg       0.98      0.99      0.98     27419
weighted avg       0.99      0.99      0.99     27419



Проверим новые значения статей, далее разметим их по универсальным категориям в новый staging-словарь.

In [17]:
data_new_staging_articles.articles__name.nunique()

48

In [18]:
data_new_staging_articles.articles__name.value_counts()

статья для удаления                   1128
5. озон банк физлица                   909
сайт донаты                            278
1.пожертвования напрямую на р/с        215
магазин                                144
тинькоф сбп                             88
вторсырье                               81
тинькофф кешбэк во благо                62
7. мтс физлица                          53
2. пожертвования через сайт фонда       52
12. мос ру платформа                    48
8. тбанк физлица                        48
17. туба физлица                        48
17.1. туба физлица сбп                  38
физ лица совкомбанк                     29
пожертвования с сайта                   27
4. пожертвования на мероприятиях        27
совкомбанк. сбп                         23
по коду (сбер)                          14
работа с юр. лицами/услуги              14
11. вк добро физлица                    14
6. втб физлица                          13
14. совкомбанк продобро                 13
3. пожертво

Загрузим staging-словарь с новыми значениями статей, раздадим истинные метки классификации и посчитаем метрики

In [52]:
staging_dict = pd.read_csv('staging_dict.csv')

In [53]:
staging_dict.tail()

,raw,universal_category,uc_id
42,помощь рядом бф,пожертвования через платформы,2
43,13. помощь рядом физлица,пожертвования через платформы,2
44,фонды и нко,пожертвования от юридических лиц (напрямую),3
45,внесение на р/сч налички,продажа товаров,6
46,"""нескучающие ручки""",продажа услуг,5


In [56]:
data_new_staging_articles.drop(columns=['target'], inplace=True)

data_new_staging_articles = data_new_staging_articles.merge(
    staging_dict[['raw', 'uc_id']],
    left_on='articles__name',
    right_on='raw',
    how='left'
)

data_new_staging_articles.rename(columns={'uc_id': 'target'}, inplace=True)
data_new_staging_articles.drop(columns=['raw'], inplace=True)
data_new_staging_articles.dropna(subset=["target"], inplace=True) # если мы какието строки не размечали спецмально, например "статья для удаления"
data_new_staging_articles.head(3)

,id,account_id,contractor_id,date,payments_amount,purpose,article_id,expenditure,project_id,counterpartie_id,donor_id,robot_id,donor_cat_id,accounts__id,accounts__user_id,articles__id,articles__user_id,articles__parent_id,articles__name,projects__id,projects__user_id,projects__parent_id,projects__name,counterparties__id,counterparties__user_id,counterparties__parent_id,counterparties__name,robots__id,robots__user_id,article_alternative_names__user_id,uc__uc_id,target
0,1207020,113,1023,2025-12-21,50.0,C302112251558957 Возм. по согл. в СБП № 9705093554 от 2020-02-27 Благотворительное пожертвование,49859,incoming,1980,0,0,12679,12647,113.0,187.0,49859.0,187.0,47651.0,5. озон банк физлица,1980.0,187.0,0.0,Уставные цели,12647.0,187.0,13621.0,Ozon банк пожертвования физлиц,12679.0,187.0,NaN,1.0,1.0
1,1207022,113,1023,2025-12-21,200.0,C302112251294979 Возм. по согл. в СБП № 9705093554 от 2020-02-27 Благотворительное пожертвование,49859,incoming,1980,0,0,12679,12647,113.0,187.0,49859.0,187.0,47651.0,5. озон банк физлица,1980.0,187.0,0.0,Уставные цели,12647.0,187.0,13621.0,Ozon банк пожертвования физлиц,12679.0,187.0,NaN,1.0,1.0
2,1207024,113,1023,2025-12-21,1.0,C302112251249987 Возм. по согл. в СБП № 9705093554 от 2020-02-27 Благотворительное пожертвование,49859,incoming,1980,0,0,12679,12647,113.0,187.0,49859.0,187.0,47651.0,5. озон банк физлица,1980.0,187.0,0.0,Уставные цели,12647.0,187.0,13621.0,Ozon банк пожертвования физлиц,12679.0,187.0,NaN,1.0,1.0


In [57]:
# считаем метрики прогнозов по статьям из staging-словаря статей (новые формулировки статей, которые модель не видела) 

accuracy = accuracy_score(data_new_staging_articles['target'], data_new_staging_articles['uc__uc_id'])
precision = precision_score(data_new_staging_articles['target'], data_new_staging_articles['uc__uc_id'], average='weighted')
recall = recall_score(data_new_staging_articles['target'], data_new_staging_articles['uc__uc_id'], average='macro')
f1 = f1_score(data_new_staging_articles['target'], data_new_staging_articles['uc__uc_id'], average='weighted')

print("Accuracy:", accuracy)
print("Precision (weighted):", precision)
print("Recall (macro):", recall)
print("F1-score (weighted):", f1)

print("\nОтчет по классам:")
print(classification_report(data_new_staging_articles['target'], data_new_staging_articles['uc__uc_id']))

Accuracy: 0.8837920489296636
Precision (weighted): 0.953704239779449
Recall (macro): 0.41364881442298895
F1-score (weighted): 0.9076134947043011

Отчет по классам:
              precision    recall  f1-score   support

         1.0       0.99      0.95      0.97      1792
         2.0       0.89      0.98      0.93       235
         3.0       0.02      0.25      0.03         4
         4.0       0.20      0.50      0.29         2
         5.0       0.01      0.07      0.01        15
         6.0       0.86      0.36      0.51       227
         7.0       0.33      0.20      0.25         5
         8.0       0.00      0.00      0.00         9

    accuracy                           0.88      2289
   macro avg       0.41      0.41      0.37      2289
weighted avg       0.95      0.88      0.91      2289



/opt/anaconda3/envs/datasphere_cli_env/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/opt/anaconda3/envs/datasphere_cli_env/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/opt/anaconda3/envs/datasphere_cli_env/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(re